In [2]:
import os

PROJECT = "/content/plp-da-a3-duka-sales-and-agent-cleanup"

os.chdir(PROJECT)

print("Current directory:", os.getcwd())
print("Available files:", os.listdir())

FileNotFoundError: [Errno 2] No such file or directory: '/content/plp-da-a3-duka-sales-and-agent-cleanup'

In [3]:
import os

print("Current directory:", os.getcwd())
print("Content directory:", os.listdir("/content"))

PROJECT = "/content/plp-da-a3-duka-sales-and-agent-cleanup"

print("Project folder exists:", os.path.exists(PROJECT))

Current directory: /content
Content directory: ['.config', 'sample_data']
Project folder exists: False


In [4]:
import os

PROJECT = "/content/plp-da-a3-duka-sales-and-agent-cleanup"

os.makedirs(PROJECT, exist_ok=True)
os.chdir(PROJECT)

print("Project created:", os.getcwd())

Project created: /content/plp-da-a3-duka-sales-and-agent-cleanup


In [5]:
!python generate_data.py


python3: can't open file '/content/plp-da-a3-duka-sales-and-agent-cleanup/generate_data.py': [Errno 2] No such file or directory


In [7]:
import os

os.makedirs(
    "/content/plp-da-a3-duka-sales-and-agent-cleanup",
    exist_ok=True
)

os.chdir("/content/plp-da-a3-duka-sales-and-agent-cleanup")

print(os.getcwd())

/content/plp-da-a3-duka-sales-and-agent-cleanup


In [8]:
%%writefile generate_data.py

Writing generate_data.py


In [9]:
!python3 generate_data.py

In [10]:
import os

print(os.listdir())

['generate_data.py']


In [11]:
!python3 generate_data.py

In [12]:
import os

print(os.listdir())

['generate_data.py']


In [14]:
!ps aux | grep '[p]ython3 generate_data.py'

In [15]:
import os

files = ["duka_sales.csv", "agents.csv", "agent_transactions.csv"]

for file in files:
    print(file, "EXISTS" if os.path.exists(file) else "NOT CREATED")

duka_sales.csv NOT CREATED
agents.csv NOT CREATED
agent_transactions.csv NOT CREATED


In [16]:
!python3 -u generate_data.py
print("Generator execution finished.")

Generator execution finished.


In [17]:
with open("generate_data.py", "r") as f:
    print(f.read())

In [18]:
import os

print("File size:", os.path.getsize("generate_data.py"), "bytes")

File size: 1 bytes


In [19]:
%%writefile generate_data.py

import numpy as np
import pandas as pd

# DUKA ELECTRONICS
rng = np.random.default_rng(2026)
n = 1000

products = {
    "Phone": 15000,
    "Laptop": 65000,
    "Headphones": 2500,
    "Tablet": 28000,
    "Charger": 1200
}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "order_date": pd.to_datetime("2026-01-01")
                  + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "branch": rng.choice(
        ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
        n, p=[0.4, 0.2, 0.15, 0.15, 0.1]
    ),
    "product": rng.choice(list(products), n),
    "quantity": rng.integers(1, 6, n),
    "channel": rng.choice(
        ["In-store", "Online", "Phone order"],
        n, p=[0.55, 0.35, 0.10]
    ),
})

sales["unit_price"] = sales["product"].map(products)
sales.to_csv("duka_sales.csv", index=False)


# MOBILE MONEY AGENT NETWORK
rng = np.random.default_rng(44)
n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(
        ["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20
    ),
    "onboarded": pd.date_range(
        "2024-01-01", periods=20, freq="MS"
    ).strftime("%Y-%m-%d"),
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(
        ["Deposit", "withdrawal", "DEPOSIT ",
         "Withdrawal", "deposit"], n
    ),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (
        pd.to_datetime("2026-03-01")
        + pd.to_timedelta(rng.integers(0, 60, n), unit="D")
    ).strftime("%Y-%m-%d"),
})

idx = rng.choice(n, 30, replace=False)

txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = (
    txns.loc[idx[25:], "txn_date"].str.replace("-", "/")
)

txns = pd.concat(
    [txns, txns.sample(12, random_state=1)],
    ignore_index=True
)

txns.loc[
    rng.choice(len(txns), 8, replace=False), "agent_id"
] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

print("All three datasets generated successfully!")

Overwriting generate_data.py


In [20]:
!python3 generate_data.py

All three datasets generated successfully!


In [21]:
import pandas as pd

print("Duka:", pd.read_csv("duka_sales.csv").shape)
print("Agents:", pd.read_csv("agents.csv").shape)
print("Transactions:", pd.read_csv("agent_transactions.csv").shape)

Duka: (1000, 7)
Agents: (20, 3)
Transactions: (612, 5)


In [22]:
import pandas as pd
import numpy as np

sales = pd.read_csv(
    "duka_sales.csv",
    parse_dates=["order_date"]
)

print("FIRST FIVE ROWS")
display(sales.head())

print("LAST FIVE ROWS")
display(sales.tail())

print("DATASET SHAPE")
print(sales.shape)

print("DATA TYPES")
sales.info()

print("STATISTICAL SUMMARY")
display(sales.describe())

print("UNIQUE VALUES")
display(sales.nunique())

print("MISSING VALUES")
display(sales.isna().sum())

print("DATE RANGE")
print(sales["order_date"].min(), "to", sales["order_date"].max())

FIRST FIVE ROWS


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


LAST FIVE ROWS


,order_id,order_date,branch,product,quantity,channel,unit_price
995,1996,2026-03-04,Mombasa,Laptop,5,In-store,65000
996,1997,2026-03-06,Nairobi,Tablet,2,Phone order,28000
997,1998,2026-01-20,Nairobi,Laptop,3,Online,65000
998,1999,2026-03-02,Nairobi,Headphones,2,In-store,2500
999,2000,2026-01-30,Nakuru,Charger,4,In-store,1200


DATASET SHAPE
(1000, 7)
DATA TYPES
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB
STATISTICAL SUMMARY


,order_id,order_date,quantity,unit_price
count,1000.000000,1000,1000.000000,1000.000000
mean,1500.500000,2026-03-30 08:44:09.600000,3.012000,23918.300000
min,1001.000000,2026-01-01 00:00:00,1.000000,1200.000000
25%,1250.750000,2026-02-14 00:00:00,2.000000,2500.000000
50%,1500.500000,2026-03-29 00:00:00,3.000000,15000.000000
75%,1750.250000,2026-05-14 00:00:00,4.000000,28000.000000
max,2000.000000,2026-06-29 00:00:00,5.000000,65000.000000
std,288.819436,NaN,1.416991,23908.790546


UNIQUE VALUES


,0
order_id,1000
order_date,179
branch,5
product,5
quantity,5
channel,3
unit_price,5


MISSING VALUES


,0
order_id,0
order_date,0
branch,0
product,0
quantity,0
channel,0
unit_price,0


DATE RANGE
2026-01-01 00:00:00 to 2026-06-29 00:00:00


In [23]:
print("BRANCH DISTRIBUTION")
display(sales["branch"].value_counts(normalize=True).mul(100).round(2))

print("CHANNEL DISTRIBUTION")
display(sales["channel"].value_counts(normalize=True).mul(100).round(2))

BRANCH DISTRIBUTION


,proportion
branch,
Nairobi,37.6
Mombasa,18.6
Nakuru,16.2
Kisumu,15.9
Eldoret,11.7


CHANNEL DISTRIBUTION


,proportion
channel,
In-store,57.9
Online,32.0
Phone order,10.1


Nairobi accounts for the largest proportion of orders (37.6%), while Eldoret has the smallest share (11.7%). In-store purchases dominate the sales channels at 57.9%, followed by online purchases at 32.0% and phone orders at 10.1%. This suggests that physical stores are Duka's primary sales channel, although online sales also represent a substantial part of the business

In [24]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]

sales["month"] = sales["order_date"].dt.month_name()

sales["weekday"] = sales["order_date"].dt.day_name()

sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5

sales["price_band"] = np.where(
    sales["unit_price"] >= 28000,
    "Premium",
    "Value"
)

display(sales.head())

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,March,Saturday,True,Value


In [25]:
branch_performance = sales.groupby("branch").agg(
    total_revenue=("revenue", "sum"),
    total_orders=("order_id", "count")
)

branch_performance = branch_performance.sort_values(
    by="total_revenue",
    ascending=False
)

display(branch_performance)

,total_revenue,total_orders
branch,,
Nairobi,26106100,376
Mombasa,14003700,186
Nakuru,11970400,162
Kisumu,10868100,159
Eldoret,8844200,117


In [26]:
print("Branch ranked by revenue:")
display(branch_performance["total_revenue"].sort_values(ascending=False))

print("Branch ranked by number of orders:")
display(branch_performance["total_orders"].sort_values(ascending=False))

Branch ranked by revenue:


,total_revenue
branch,
Nairobi,26106100
Mombasa,14003700
Nakuru,11970400
Kisumu,10868100
Eldoret,8844200


Branch ranked by number of orders:


,total_orders
branch,
Nairobi,376
Mombasa,186
Nakuru,162
Kisumu,159
Eldoret,117


Nairobi is the best-performing branch, generating the highest revenue of KES 26,106,100 from 376 orders. Eldoret is the lowest-performing branch, generating KES 8,844,200 from 117 orders. The revenue and order-volume rankings are identical, suggesting that branches with more orders also generate higher total revenue. Duka should investigate Nairobi's successful sales practices and identify ways to improve performance in Eldoret.

In [27]:
product_performance = sales.groupby("product").agg(
    total_revenue=("revenue", "sum"),
    units_sold=("quantity", "sum")
)

print("TOP 3 PRODUCTS BY REVENUE")
display(
    product_performance
    .sort_values("total_revenue", ascending=False)
    .head(3)
)

print("TOP 3 PRODUCTS BY UNITS SOLD")
display(
    product_performance
    .sort_values("units_sold", ascending=False)
    .head(3)
)

TOP 3 PRODUCTS BY REVENUE


,total_revenue,units_sold
product,,
Laptop,41535000,639
Tablet,19348000,691
Phone,8880000,592


TOP 3 PRODUCTS BY UNITS SOLD


,total_revenue,units_sold
product,,
Tablet,19348000,691
Laptop,41535000,639
Phone,8880000,592


Laptops generate the highest revenue (KES 41,535,000) despite selling only 639 units. Tablets have the highest sales volume, with 691 units sold, generating KES 19,348,000. Phones rank third in both revenue and units sold. This shows that high-priced products can generate more revenue even when fewer units are sold. Duka should maintain laptop availability while continuing to promote tablets due to their high sales volume.

In [28]:
weekend_analysis = sales.groupby("is_weekend").agg(
    total_orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    average_revenue=("revenue", "mean")
)

weekend_analysis.index = ["Weekdays", "Weekends"]

display(weekend_analysis.round(2))

,total_orders,total_revenue,average_revenue
Weekdays,723,51856200,71723.65
Weekends,277,19936300,71972.20


Weekdays generate higher total revenue (KES 51,856,200) than weekends (KES 19,936,300), mainly because they account for more orders (723 compared with 277). However, weekends have a slightly higher average revenue per order (KES 71,972.20) than weekdays (KES 71,723.65). Duka could explore weekend promotions to increase order volume while maintaining the higher average order value.

In [29]:
monthly_sales = sales.groupby("month").agg(
    total_revenue=("revenue", "sum"),
    total_orders=("order_id", "count")
)

# Arrange months in calendar order
month_order = [
    "January", "February", "March",
    "April", "May", "June"
]

monthly_sales = monthly_sales.reindex(month_order)

display(monthly_sales)

best_month = monthly_sales["total_revenue"].idxmax()
worst_month = monthly_sales["total_revenue"].idxmin()

best_revenue = monthly_sales["total_revenue"].max()
worst_revenue = monthly_sales["total_revenue"].min()

percentage_difference = (
    (best_revenue - worst_revenue) / worst_revenue
) * 100

print("Best month:", best_month)
print("Worst month:", worst_month)
print("Percentage by which the best month exceeds the worst month:",
      round(percentage_difference, 2), "%")

,total_revenue,total_orders
month,,
January,11786600,173
February,11772000,159
March,12396800,186
April,13518300,167
May,11415400,153
June,10903400,162


Best month: April
Worst month: June
Percentage by which the best month exceeds the worst month: 23.98 %


April was the best-performing month, generating KES 13,518,300 in revenue from 167 orders. June was the weakest month, generating KES 10,903,400 from 162 orders. April's revenue was 23.98% higher than June's. Duka should investigate the factors behind April's strong performance and consider strategies to improve sales during June.

In [30]:
premium_online = sales[
    (sales["price_band"] == "Premium") &
    (sales["channel"] == "Online") &
    (sales["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

print("Number of qualifying orders:", len(premium_online))

print("Total revenue:", premium_online["revenue"].sum())

display(premium_online.head())

Number of qualifying orders: 83
Total revenue: 10218000


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
6,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,January,Thursday,False,Premium
13,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,June,Friday,False,Premium
32,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,April,Wednesday,False,Premium
35,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,March,Monday,False,Premium
39,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,February,Tuesday,False,Premium


In [31]:
import os

os.makedirs("outputs", exist_ok=True)

premium_online.to_csv(
    "outputs/premium_online_coast_capital.csv",
    index=False
)

print("File exported successfully!")

File exported successfully!


A total of 83 orders met the filtering criteria, generating KES 10,218,000 in revenue. These orders involved Premium products purchased online from the Nairobi and Mombasa branches. This segment represents an opportunity for Duka to strengthen its online sales strategy and target customers interested in higher-priced products.

In [32]:
import os
import pandas as pd

file_path = "outputs/premium_online_coast_capital.csv"

print("File exists:", os.path.exists(file_path))

exported = pd.read_csv(file_path)

print("Exported shape:", exported.shape)
print("Exported revenue:", exported["revenue"].sum())

File exists: True
Exported shape: (83, 12)
Exported revenue: 10218000


Executive Summary
1. Duka Electronics recorded 1,000 orders across five branches between January and June 2026.
2. Nairobi was the best-performing branch, generating KES 26,106,100 from 376 orders.
3. Laptops generated the highest product revenue of KES 41,535,000, while tablets recorded the highest sales volume at 691 units.
4. April was the strongest month, generating KES 13,518,300, which was 23.98% higher than June, the weakest month.
5. A total of 83 Premium online orders from Nairobi and Mombasa generated KES 10,218,000 in revenue.

In [33]:
sales.isnull().sum()

,0
order_id,0
order_date,0
branch,0
product,0
quantity,0
channel,0
unit_price,0
revenue,0
month,0
weekday,0


Data Quality Observation
The dataset contains 1,000 records and 12 columns, with zero missing values across all columns. This indicates that the dataset is complete and does not require missing-value treatment before analysis. The dataset covers five branches and three sales channels over the period January to June 2026.

In [34]:
from google.colab import files
files.download("duka_sales.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
sales.to_csv("duka_sales_enriched.csv", index=False)


In [36]:
from google.colab import files
files.download("duka_sales_enriched.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [37]:
import os
os.makedirs("excel", exist_ok=True)

In [38]:
from google.colab import files
uploaded = files.upload()

Saving Duka Branch Summary.xlsx to Duka Branch Summary.xlsx


In [39]:
import shutil

filename = next(iter(uploaded))
shutil.move(filename, "excel/duka_branch_summary.xlsx")

'excel/duka_branch_summary.xlsx'

In [40]:
import os

path = "excel/duka_branch_summary.xlsx"

print("Workbook exists:", os.path.exists(path))
if os.path.exists(path):
    print("File size:", os.path.getsize(path), "bytes")

Workbook exists: True
File size: 73401 bytes


In [41]:
import os

print("Current folder:", os.getcwd())
print("\nProject files:")

for item in os.listdir():
    print(item)


Current folder: /content/plp-da-a3-duka-sales-and-agent-cleanup

Project files:
generate_data.py
agent_transactions.csv
excel
duka_sales_enriched.csv
duka_sales.csv
outputs
agents.csv


In [42]:
import pandas as pd
import numpy as np

agents = pd.read_csv("agents.csv")
txns = pd.read_csv("agent_transactions.csv")

print("AGENT REGISTER")
display(agents.head())

print("TRANSACTION DATA")
display(txns.head())

AGENT REGISTER


,agent_id,county,onboarded
0,AG001,Mombasa,2024-01-01
1,AG002,Nairobi,2024-02-01
2,AG003,Kisumu,2024-03-01
3,AG004,Kiambu,2024-04-01
4,AG005,Mombasa,2024-05-01


TRANSACTION DATA


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


In [43]:
print("AGENT REGISTER SHAPE:", agents.shape)
print("TRANSACTION SHAPE:", txns.shape)

print("\nAGENT REGISTER INFORMATION")
agents.info()

print("\nTRANSACTION INFORMATION")
txns.info()

print("\nTRANSACTION SUMMARY")
display(txns.describe(include="all"))

AGENT REGISTER SHAPE: (20, 3)
TRANSACTION SHAPE: (612, 5)

AGENT REGISTER INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   agent_id   20 non-null     object
 1   county     20 non-null     object
 2   onboarded  20 non-null     object
dtypes: object(3)
memory usage: 612.0+ bytes

TRANSACTION INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   txn_id    612 non-null    object 
 1   agent_id  612 non-null    object 
 2   txn_type  612 non-null    object 
 3   amount    590 non-null    float64
 4   txn_date  612 non-null    object 
dtypes: float64(1), object(4)
memory usage: 24.0+ KB

TRANSACTION SUMMARY


,txn_id,agent_id,txn_type,amount,txn_date
count,612,612,612,590.000000,612
unique,600,21,5,NaN,65
top,TX00330,AG017,deposit,NaN,2026-04-23
freq,2,38,142,NaN,21
mean,NaN,NaN,NaN,4537.201695,NaN
std,NaN,NaN,NaN,22836.165375,NaN
min,NaN,NaN,NaN,109.000000,NaN
25%,NaN,NaN,NaN,1086.750000,NaN
50%,NaN,NaN,NaN,1779.000000,NaN
75%,NaN,NaN,NaN,3069.750000,NaN


In [44]:
print("MISSING VALUES")
display(txns.isna().sum())

print("\nDUPLICATE ROWS")
print(txns.duplicated().sum())

print("\nDUPLICATE TRANSACTION IDs")
print(txns["txn_id"].duplicated().sum())

print("\nTRANSACTION TYPES")
display(txns["txn_type"].value_counts(dropna=False))

print("\nAGENT IDs")
display(txns["agent_id"].value_counts().head(10))

MISSING VALUES


,0
txn_id,0
agent_id,0
txn_type,0
amount,22
txn_date,0



DUPLICATE ROWS
11

DUPLICATE TRANSACTION IDs
12

TRANSACTION TYPES


,count
txn_type,
deposit,142
withdrawal,131
Deposit,128
DEPOSIT,111
Withdrawal,100



AGENT IDs


,count
agent_id,
AG017,38
AG006,38
AG004,37
AG007,36
AG010,36
AG003,35
AG016,34
AG014,33
AG015,29


The transaction dataset contains 612 records and five columns, while the agent register contains 20 agents and three columns. The transaction dataset has 12 duplicate transaction IDs, 11 exact duplicate rows and 22 missing transaction amounts. Transaction types have five inconsistent representations that need standardization. The maximum transaction amount is KES 250,000, which requires outlier investigation. The initial inspection also indicates inconsistent date formatting and potentially unregistered agent references that require further validation.

In [45]:
# 1. Check unusual transaction dates
print("DATE FORMATS")
display(txns["txn_date"].value_counts().head(15))

# 2. Check unregistered agents
unknown_agents = set(txns["agent_id"]) - set(agents["agent_id"])

print("\nUNKNOWN AGENT IDs:")
print(unknown_agents)

print("\nTRANSACTIONS WITH UNKNOWN AGENTS:")
print(txns["agent_id"].isin(unknown_agents).sum())

# 3. Check unusually large amounts
print("\nTRANSACTIONS WITH AMOUNT 250000:")
display(txns[txns["amount"] == 250000])

# 4. Examine duplicate transaction IDs
print("\nDUPLICATE TRANSACTION RECORDS:")
display(
    txns[txns.duplicated("txn_id", keep=False)]
    .sort_values("txn_id")
)

DATE FORMATS


,count
txn_date,
2026-04-23,21
2026-04-24,19
2026-04-06,18
2026-03-09,18
2026-04-28,16
2026-03-03,15
2026-04-20,14
2026-03-19,14
2026-04-01,14



UNKNOWN AGENT IDs:
{'AG099'}

TRANSACTIONS WITH UNKNOWN AGENTS:
8

TRANSACTIONS WITH AMOUNT 250000:


,txn_id,agent_id,txn_type,amount,txn_date
306,TX00307,AG012,deposit,250000.0,2026-03-02
315,TX00316,AG014,deposit,250000.0,2026-04-01
424,TX00425,AG005,Withdrawal,250000.0,2026-03-02
463,TX00464,AG009,Withdrawal,250000.0,2026-04-11
594,TX00595,AG016,Deposit,250000.0,2026-03-11



DUPLICATE TRANSACTION RECORDS:


,txn_id,agent_id,txn_type,amount,txn_date
23,TX00024,AG017,DEPOSIT,2798.0,2026-04-28
605,TX00024,AG017,DEPOSIT,2798.0,2026-04-28
66,TX00067,AG002,deposit,NaN,2026-04-07
608,TX00067,AG002,deposit,NaN,2026-04-07
165,TX00166,AG010,deposit,4146.0,2026-04-18
609,TX00166,AG010,deposit,4146.0,2026-04-18
201,TX00202,AG010,Deposit,2243.0,2026-03-22
604,TX00202,AG010,Deposit,2243.0,2026-03-22
611,TX00330,AG016,withdrawal,1219.0,2026-04-07
329,TX00330,AG016,withdrawal,1219.0,2026-04-07


In [46]:

%%writefile cleaning_pipeline.py

import pandas as pd


def clean_agent_data(txns, agents):
    """
    Clean agent transactions, record data-quality issues,
    and merge agent county information.

    Returns:
        clean_df: cleaned transaction DataFrame
        log: data-quality audit DataFrame
    """

    clean_df = txns.copy()
    audit = []

    def add_log(issue, dimension, column, rows_affected):
        audit.append({
            "issue": issue,
            "dimension": dimension,
            "column": column,
            "rows_affected": int(rows_affected)
        })

    # 1. Remove duplicate transaction IDs, keeping the first record
    duplicate_count = clean_df["txn_id"].duplicated().sum()
    clean_df = clean_df.drop_duplicates(
        subset="txn_id", keep="first"
    ).copy()

    add_log(
        "Duplicate transaction IDs removed",
        "Uniqueness",
        "txn_id",
        duplicate_count
    )

    # 2. Convert amounts to numeric values
    clean_df["amount"] = pd.to_numeric(
        clean_df["amount"], errors="coerce"
    )

    # 3. Flag missing amounts before imputing them
    clean_df["amount_imputed"] = clean_df["amount"].isna()
    missing_count = clean_df["amount_imputed"].sum()

    # 4. Detect outliers using the IQR method
    q1 = clean_df["amount"].quantile(0.25)
    q3 = clean_df["amount"].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    clean_df["is_outlier"] = (
        (clean_df["amount"] < lower_bound) |
        (clean_df["amount"] > upper_bound)
    )

    outlier_count = clean_df["is_outlier"].sum()

    # 5. Replace missing amounts with the median
    median_amount = clean_df["amount"].median()
    clean_df["amount"] = clean_df["amount"].fillna(median_amount)

    add_log(
        "Missing amounts filled with the median",
        "Completeness",
        "amount",
        missing_count
    )

    add_log(
        "Unusual amounts flagged using the IQR method",
        "Accuracy",
        "amount",
        outlier_count
    )

    # 6. Standardize transaction types
    original_types = clean_df["txn_type"].copy()

    clean_df["txn_type"] = (
        clean_df["txn_type"].astype("string").str.strip().str.title()
    )

    standardized_count = (
        original_types != clean_df["txn_type"]
    ).sum()

    add_log(
        "Transaction types standardized",
        "Consistency",
        "txn_type",
        standardized_count
    )

    # 7. Standardize transaction dates
    original_dates = clean_df["txn_date"].astype("string")

    clean_df["txn_date"] = pd.to_datetime(
        original_dates.str.replace("/", "-", regex=False),
        errors="coerce"
    )

    invalid_dates = clean_df["txn_date"].isna().sum()

    clean_df["txn_date"] = clean_df["txn_date"].dt.strftime("%Y-%m-%d")

    changed_dates = (
        original_dates != clean_df["txn_date"].astype("string")
    ).sum()

    add_log(
        "Transaction dates standardized to ISO format",
        "Consistency",
        "txn_date",
        changed_dates
    )

    add_log(
        "Invalid transaction dates",
        "Validity",
        "txn_date",
        invalid_dates
    )

    # 8. Merge agent county information
    clean_df = clean_df.merge(
        agents[["agent_id", "county"]],
        on="agent_id",
        how="left",
        validate="many_to_one"
    )

    unknown_count = clean_df["county"].isna().sum()

    clean_df["county"] = clean_df["county"].fillna("Unknown")

    add_log(
        "Transactions linked to unregistered agents",
        "Validity",
        "agent_id",
        unknown_count
    )

    # 9. Create the audit log
    log = pd.DataFrame(
        audit,
        columns=["issue", "dimension", "column", "rows_affected"]
    )

    return clean_df, log


Writing cleaning_pipeline.py


In [47]:
from cleaning_pipeline import clean_agent_data

clean_df, audit_log = clean_agent_data(txns, agents)

print("CLEANED DATA:")
print(clean_df.shape)
display(clean_df.head())

print("\nAUDIT LOG:")
display(audit_log)

CLEANED DATA:
(600, 8)


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru



AUDIT LOG:


,issue,dimension,column,rows_affected
0,Duplicate transaction IDs removed,Uniqueness,txn_id,12
1,Missing amounts filled with the median,Completeness,amount,20
2,Unusual amounts flagged using the IQR method,Accuracy,amount,40
3,Transaction types standardized,Consistency,txn_type,376
4,Transaction dates standardized to ISO format,Consistency,txn_date,5
5,Invalid transaction dates,Validity,txn_date,0
6,Transactions linked to unregistered agents,Validity,agent_id,7


In [48]:
print("CLEANED DATA VALIDATION")
print("-----------------------")

print("Shape:", clean_df.shape)
print("Unique transaction IDs:", clean_df["txn_id"].nunique())
print("Duplicate transaction IDs:", clean_df["txn_id"].duplicated().sum())

print("\nTransaction types:")
print(clean_df["txn_type"].value_counts())

print("\nMissing values:")
print(clean_df.isna().sum())

print("\nAmount data type:", clean_df["amount"].dtype)
print("Date format examples:")
print(clean_df["txn_date"].head())

print("\nImputed amounts:", clean_df["amount_imputed"].sum())
print("Flagged outliers:", clean_df["is_outlier"].sum())
print("Unknown agents:", (clean_df["county"] == "Unknown").sum())

print("\nAUDIT LOG:")
display(audit_log)

CLEANED DATA VALIDATION
-----------------------
Shape: (600, 8)
Unique transaction IDs: 600
Duplicate transaction IDs: 0

Transaction types:
txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: Int64

Missing values:
txn_id            0
agent_id          0
txn_type          0
amount            0
txn_date          0
amount_imputed    0
is_outlier        0
county            0
dtype: int64

Amount data type: float64
Date format examples:
0    2026-03-03
1    2026-04-27
2    2026-04-29
3    2026-04-24
4    2026-04-17
Name: txn_date, dtype: object

Imputed amounts: 20
Flagged outliers: 40
Unknown agents: 7

AUDIT LOG:


,issue,dimension,column,rows_affected
0,Duplicate transaction IDs removed,Uniqueness,txn_id,12
1,Missing amounts filled with the median,Completeness,amount,20
2,Unusual amounts flagged using the IQR method,Accuracy,amount,40
3,Transaction types standardized,Consistency,txn_type,376
4,Transaction dates standardized to ISO format,Consistency,txn_date,5
5,Invalid transaction dates,Validity,txn_date,0
6,Transactions linked to unregistered agents,Validity,agent_id,7


In [49]:
county_summary = clean_df.groupby(
    ["county", "txn_type"]
).agg(
    transaction_count=("txn_id", "count"),
    total_amount=("amount", "sum"),
    average_amount=("amount", "mean")
).reset_index()

display(county_summary)

,county,txn_type,transaction_count,total_amount,average_amount
0,Kiambu,Deposit,43,334737.0,7784.581395
1,Kiambu,Withdrawal,27,77050.5,2853.722222
2,Kisumu,Deposit,74,436420.5,5897.574324
3,Kisumu,Withdrawal,37,76374.5,2064.175676
4,Mombasa,Deposit,72,148986.0,2069.250000
5,Mombasa,Withdrawal,61,368592.0,6042.491803
6,Nairobi,Deposit,118,543590.0,4606.694915
7,Nairobi,Withdrawal,62,162036.0,2613.483871
8,Nakuru,Deposit,60,184890.0,3081.500000
9,Nakuru,Withdrawal,39,328594.5,8425.500000


In [50]:
clean_df["txn_date"] = pd.to_datetime(clean_df["txn_date"])

clean_df["week"] = clean_df["txn_date"].dt.to_period("W").astype(str)

weekly_pivot = clean_df.pivot_table(
    index="county",
    columns="week",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

display(weekly_pivot)

week,2026-02-23/2026-03-01,2026-03-02/2026-03-08,2026-03-09/2026-03-15,2026-03-16/2026-03-22,2026-03-23/2026-03-29,2026-03-30/2026-04-05,2026-04-06/2026-04-12,2026-04-13/2026-04-19,2026-04-20/2026-04-26,2026-04-27/2026-05-03
county,,,,,,,,,,
Kiambu,0.0,18317.0,267771.5,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0
Kisumu,1893.0,287933.0,23186.0,28125.0,27077.0,41077.0,15341.0,29416.0,37439.0,21308.0
Mombasa,6759.0,285669.5,30477.5,25144.5,17050.0,38344.0,47092.5,26730.0,16687.0,23624.0
Nairobi,7046.0,68967.0,52860.5,43107.5,41373.5,282931.0,56786.5,56432.5,65187.0,30934.5
Nakuru,2743.5,56574.0,28730.0,39795.0,18998.5,18145.0,276609.0,26152.5,30862.0,14875.0
Unknown,0.0,0.0,0.0,1639.0,0.0,8507.0,9824.0,0.0,1877.0,5309.0


In [51]:
import os

os.makedirs("outputs", exist_ok=True)

clean_df.to_csv(
    "outputs/agent_transactions_clean.csv",
    index=False
)

print("File saved successfully!")
print("Shape:", clean_df.shape)
print("File exists:", os.path.exists(
    "outputs/agent_transactions_clean.csv"
))

File saved successfully!
Shape: (600, 9)
File exists: True


# Data Quality Report

## 1. Completeness
The original dataset contained 20 missing transaction amounts after duplicate removal. These values were replaced with the median amount, and all missing values were resolved.

## 2. Uniqueness
The dataset contained 12 duplicate transaction IDs. These were removed by retaining the first occurrence of each transaction ID. The cleaned dataset contains 600 unique transactions.

## 3. Consistency
A total of 376 transaction types were standardized to either `Deposit` or `Withdrawal`. Five transaction dates were converted to the ISO format (`YYYY-MM-DD`).

## 4. Validity
Eight transactions were associated with an unregistered agent ID (`AG099`). These records were retained and their county was labelled `Unknown`. No invalid transaction dates were found.

## 5. Accuracy
The IQR method identified 40 unusually high or low transaction amounts. These values were flagged for further investigation rather than deleted. Outliers are not necessarily errors.

## 6. Timeliness
The transaction records cover March and April 2026. Dates were standardized to support consistent chronological and weekly analysis.

## Data Privacy and Protection
Customer phone numbers are personal data under Kenya's Data Protection Act, 2019. If collected, they should be protected through encryption, restricted access, data minimization, secure storage and appropriate retention policies. Reports should use aggregated or anonymized information wherever possible.